# Lecture 3 — Computing and verifying a Nash equilibrium

A practical companion to the homotopy continuation method explained in [Lecture 3](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/03-computing-nash-equilibrium.html). Run the cells in order to construct the three-player example, solve it with [LogitNash.jl](https://github.com/votroto/LogitNash.jl), and verify the result independently.

In Google Colab, choose **Runtime → Change runtime type → Julia** with a CPU runtime, then save a copy if you want to keep your changes. Julia **1.8 or later** is sufficient; no local installation is needed.


## 1. Set up the session

Install the registered package in a temporary project. The cell prints the Julia and package versions for reproducibility. If a different LogitNash version is already loaded, restart the runtime first.


In [ ]:
import Pkg
Pkg.activate(mktempdir())
Pkg.add("LogitNash")
using LogitNash, LinearAlgebra, Printf

println("Julia version: ", VERSION)
Pkg.status(["LogitNash"])


## 2. Construct the game

Use the **asymmetric cyclic matching pennies** example from [Lecture 3](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/03-computing-nash-equilibrium.html#julia-demonstration), initially with $M=9$. Each of the three payoff tensors has dimensions $2\times2\times2$; strategy indices 1 and 2 mean $H$ and $T$. The code uses the lecture's normalized payoffs.

All reported deviation gains use these normalized units; multiply by $M$ to obtain the original units. The known equilibrium is used only to check the result, not as input to the solver.


In [ ]:
function cyclic_game(M::Real)
    @assert isfinite(M) && M >= 1
    payoffs = ntuple(_ -> zeros(Float64, 2, 2, 2), 3)
    # Tensor index s = (s₁, s₂, s₃), with strategy 1 = H and strategy 2 = T.
    for s in CartesianIndices((2, 2, 2)), i in 1:3
        predecessor = mod1(i - 1, 3)
        payoffs[i][s] = if s[i] == s[predecessor]
            0.0
        elseif s[i] == 1
            1.0 / M
        else
            1.0
        end
    end
    payoffs
end

M = 9.0                         # Change this parameter in the experiments.
game = cyclic_game(M)
exact_profile = ntuple(_ -> [1/(M+1), M/(M+1)], 3)
println("Known equilibrium, each player: ", exact_profile[1], "  [H, T]")

## 3. Prepare an independent check

The checker validates the probabilities and evaluates every pure-strategy deviation directly from the payoff tensors. It returns the individual gains and their maximum as `max_gain`. First test it on the known equilibrium and the uniform profile.


In [ ]:
function check_profile(payoffs, profile; probability_tol=1e-10)
    dims = size(first(payoffs))
    n = length(payoffs)
    @assert length(dims) == n == length(profile)
    @assert all(size(u) == dims && all(isfinite, u) for u in payoffs)
    for i in 1:n
        @assert length(profile[i]) == dims[i]
        @assert all(isfinite, profile[i])
        @assert all(x -> 0 <= x <= 1, profile[i])
        @assert isapprox(sum(profile[i]), 1.0; atol=probability_tol, rtol=0)
    end
    action_values = [zeros(Float64, dims[i]) for i in 1:n]
    for s in CartesianIndices(dims), i in 1:n
        opponents_probability = prod(profile[j][s[j]] for j in 1:n if j != i)
        action_values[i][s[i]] += payoffs[i][s] * opponents_probability
    end
    expected = [dot(profile[i], action_values[i]) for i in 1:n]
    gains = [maximum(action_values[i]) - expected[i] for i in 1:n]
    (; action_values, expected, gains, max_gain=maximum(gains))
end

exact_check = check_profile(game, exact_profile)
uniform_profile = ntuple(_ -> [0.5, 0.5], 3)
uniform_check = check_profile(game, uniform_profile)

@assert abs(exact_check.max_gain) <= 1e-12
@assert isapprox(uniform_check.max_gain, (1 - 1/M)/4; atol=1e-12)
println("Known equilibrium: maximum gain = ", exact_check.max_gain)
println("Uniform profile:   maximum gain = ", uniform_check.max_gain)

## 4. Compute and verify an equilibrium

Request maximum deviation incentive at most $10^{-6}$. The iteration and precision limits allow a longer run; check `status` if the requested accuracy is not reached. Compare the maximum deviation incentive in `status.regret` with the independent calculation.


In [ ]:
epsilon_target = 1e-6
profile, status = LogitNash.solve(game;
    stop_eps=epsilon_target, stop_iters=5000, stop_lambda=1e10)
checked = check_profile(game, profile)

println("Solver status: ", status)
@printf("%8s %13s %13s %15s\n", "Player", "P(H)", "P(T)", "Deviation gain")
for i in 1:3
    @printf("%8d %13.9f %13.9f %15.6e\n",
            i, profile[i][1], profile[i][2], checked.gains[i])
end
@printf("Maximum deviation gain: %.6e\n", checked.max_gain)
@printf("Largest probability error against known NE: %.6e\n",
        maximum(maximum(abs.(profile[i] .- exact_profile[i])) for i in 1:3))

@assert isapprox(status.regret, checked.max_gain; atol=1e-12, rtol=1e-6)
@assert checked.max_gain <= epsilon_target "Requested accuracy not achieved; inspect status."
println("PASS: returned profile is a numerically verified ε-NE at the requested tolerance.")


## 5. Compare with the scalar QRE path

Compute the scalar QRE from the [equation in Lecture 3](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/03-computing-nash-equilibrium.html#eq-computing-cyclic-qre) by bisection, then plot it. These points are an independent reference, not recorded solver iterations. The table shows the maximum deviation incentive and the logit response residual at each precision.


In [ ]:
function logit_response(values, precision)
    @assert isfinite(precision) && precision >= 0
    @assert !isempty(values) && all(isfinite, values)
    weights = exp.(precision .* (values .- maximum(values)))
    weights ./ sum(weights)
end


function symmetric_qre(M, precision)
    @assert isfinite(M) && M >= 1
    @assert isfinite(precision) && precision >= 0
    (precision == 0 || M == 1) && return 0.5
    lo, hi = 1/(M+1), 0.5
    for _ in 1:100
        mid = (lo + hi)/2
        (mid == lo || mid == hi) && break
        residual = log(mid) - log1p(-mid) - precision*((1-mid)/M - mid)
        if residual > 0
            hi = mid
        else
            lo = mid
        end
    end
    (lo + hi)/2
end

precisions = [0.0, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5, 1e6]
@printf("%12s %14s %18s %18s\n", "λ", "P(H)", "max deviation gain", "QRE residual")
for precision in precisions
    x = symmetric_qre(M, precision)
    qre_profile = ntuple(_ -> [x, 1-x], 3)
    result = check_profile(game, qre_profile)
    response = logit_response([(1-x)/M, x], precision)
    residual = maximum(abs.(response .- [x, 1-x]))
    @printf("%12.1f %14.9f %18.6e %18.6e\n", precision, x, result.max_gain, residual)
    @assert residual <= 1e-8
end


In [ ]:
#| label: fig-logit-path
#| fig-cap: "The symmetric QRE approaches the Nash equilibrium as precision increases."
precision_grid = 10.0 .^ range(-2, 6; length=200)
qre_probabilities = [symmetric_qre(M, z) for z in precision_grid]
qre_gains = [check_profile(game, ntuple(_ -> [x, 1-x], 3)).max_gain
            for x in qre_probabilities]

# A self-contained SVG plot avoids a plotting-package installation.
function qre_figure(precision_grid, probabilities, gains, M)
    io = IOBuffer()
    print(io, """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 370"
        role="img" aria-label="Symmetric QRE probability and maximum deviation incentive versus precision"
        style="max-width:100%;height:auto;background:white;color:#182230">
        <rect width="960" height="370" fill="white"/>
        <style>text{font:13px sans-serif;fill:#182230}.grid{stroke:#e4e7ec}</style>
        <text x="480" y="23" text-anchor="middle" style="font-size:17px">Asymmetric cyclic matching pennies: M = $M</text>""")
    panels = ((probabilities, 0.5, "Probability of H"),
              (max.(gains, 0.0), max(0.01, maximum(gains)*1.05), "Maximum deviation incentive"))
    for (panel, (values, ymax, label)) in enumerate(panels)
        left, top, width, height = 65 + 475*(panel-1), 60, 355, 235
        xcoord(z) = left + width*(log10(z)+2)/8
        ycoord(y) = top + height*(1-y/ymax)
        print(io, "<text x='$(left+width/2)' y='45' text-anchor='middle'>$label</text>")
        for exponent in -2:2:6
            x = xcoord(10.0^exponent)
            print(io, "<line class='grid' x1='$x' y1='$top' x2='$x' y2='$(top+height)'/>")
            print(io, "<text x='$x' y='$(top+height+20)' text-anchor='middle'>10<tspan dy='-5' font-size='10'>$exponent</tspan></text>")
        end
        for tick in range(0, ymax; length=5)
            y = ycoord(tick)
            print(io, "<line class='grid' x1='$left' y1='$y' x2='$(left+width)' y2='$y'/>")
            print(io, "<text x='$(left-9)' y='$(y+4)' text-anchor='end'>", @sprintf("%.3f", tick), "</text>")
        end
        print(io, "<rect x='$left' y='$top' width='$width' height='$height' fill='none' stroke='#667085'/>")
        points = join(("$(xcoord(z)),$(ycoord(y))" for (z,y) in zip(precision_grid, values)), " ")
        print(io, "<polyline points='$points' fill='none' stroke='#175cd3' stroke-width='2.5'/>")
        if panel == 1
            y = ycoord(1/(M+1))
            print(io, "<line x1='$left' y1='$y' x2='$(left+width)' y2='$y' stroke='#b54708' stroke-width='2' stroke-dasharray='6 4'/>")
            print(io, "<text x='$(left+width-5)' y='$(y-8)' text-anchor='end' style='fill:#b54708'>Nash equilibrium</text>")
        end
        print(io, "<text x='$(left+width/2)' y='345' text-anchor='middle'>Precision λ (logarithmic scale)</text>")
    end
    print(io, "</svg>")
    HTML(String(take!(io)))
end

display(qre_figure(precision_grid, qre_probabilities, qre_gains, M))


Compare the solver's endpoint with the scalar QRE at the same precision, `status.lambda`. This comparison is specific to this example; the deviation check above applies to any game supplied as payoff tensors.


In [ ]:
x_reference = symmetric_qre(M, status.lambda)
endpoint_error = maximum(abs(profile[i][1] - x_reference) for i in 1:3)
qre_residual = maximum(
    maximum(abs.(logit_response(checked.action_values[i], status.lambda) .- profile[i]))
    for i in 1:3)
println("Scalar QRE probability at the solver's λ: ", x_reference)
println("Largest difference from scalar QRE:       ", endpoint_error)
println("Largest logit fixed-point residual:       ", qre_residual)
println("Nash deviation gain:                     ", checked.max_gain)


## 6. Optional experiments

1. Compare $M=1,3,9,99$. Record the probability of $H$, maximum deviation incentive, and iteration count.
2. Fix $M=9$ and compare requested tolerances $10^{-3},10^{-6},10^{-8}$. Record the achieved accuracy and probability error against the known equilibrium.
3. For $M=9$, convert the reported deviation incentive to original payoff units. What normalized tolerance gives an original-unit target of $10^{-6}$?

The following cells run experiments 1 and 2 and check each returned profile.


In [ ]:
function solve_and_check(M; tolerance=1e-6)
    payoffs = cyclic_game(M)
    p, info = LogitNash.solve(payoffs;
        stop_eps=tolerance, stop_iters=5000, stop_lambda=1e10)
    result = check_profile(payoffs, p)
    probability_error = maximum(abs(p[i][1] - 1/(M+1)) for i in 1:3)
    (; M, tolerance, probability=p[1][1], probability_error,
       achieved=result.max_gain, iterations=info.iteration,
       passed=result.max_gain <= tolerance, status=info)
end

@printf("%6s %12s %14s %14s %8s %8s\n", "M", "P₁(H)", "max gain", "prob. error", "steps", "passes")
for asymmetry in (1.0, 3.0, 9.0, 99.0)
    row = solve_and_check(asymmetry)
    @printf("%6.0f %12.8f %14.4e %14.4e %8d %8s\n",
        row.M, row.probability, row.achieved, row.probability_error,
        row.iterations, string(row.passed))
    @assert row.passed "Inspect the returned solver status: $(row.status)"
end


In [ ]:
@printf("%12s %14s %14s %8s %8s\n", "target", "max gain", "prob. error", "steps", "passes")
for tolerance in (1e-3, 1e-6, 1e-8)
    row = solve_and_check(9.0; tolerance=tolerance)
    @printf("%12.1e %14.4e %14.4e %8d %8s\n",
        row.tolerance, row.achieved, row.probability_error,
        row.iterations, string(row.passed))
    @assert row.passed "Inspect the returned solver status: $(row.status)"
end